# Parte 3 — Recommendations Engine

## 1. Objetivo e abordagem

O objetivo desta etapa é construir um sistema capaz de analisar as características de uma campanha de vídeo e sugerir modificações acionáveis, considerando os padrões de performance observados no dataset.

A solução utiliza uma abordagem baseada em **comparações de medianas por plataforma e atributo do criativo**, em vez de prever diretamente os resultados de alterações individuais.

Foram consideradas cinco métricas de performance: CTR, ROAS, taxa de engajamento, taxa de conversão e proporção assistida do vídeo.

**Decisões metodológicas:**
- Comparar campanhas dentro da mesma plataforma, considerando diferenças de comportamento entre Meta, TikTok e LinkedIn.
- Exigir pelo menos 10 observações válidas por grupo comparado.
- Considerar recomendações com diferenças positivas de pelo menos 10% entre as medianas.
- Priorizar a métrica associada ao objetivo da campanha durante a seleção.
- Limitar a saída a cinco recomendações, evitando alterações repetidas sobre o mesmo atributo.

As recomendações representam **associações observacionais**, não estimativas causais. Portanto, as diferenças percentuais apresentadas não devem ser interpretadas como ganhos garantidos após a aplicação das sugestões.

In [1]:
import numpy as np
import joblib

import pandas as pd
pd.set_option("display.max_columns", None)

In [2]:
dados = pd.read_csv('../data/klike_challenge_dataset.csv')
dados.head(5)

,campaign_id,date,platform,category,objective,target_audience_age,is_retargeting,video_duration_s,format,has_subtitle,has_cta,has_hook,has_face,text_density,music_voice_ratio,impressions,clicks,ctr,cpc,spend,conversions,revenue,roas,avg_watch_time_s,engagement_rate,klike_score
0,KLK-0001,2024-01-01,Meta,Branding,awareness,18-24,False,NaN,vertical,True,True,False,True,medium,0.39,150825,5882,0.0390,0.82,4823.24,48,0.00,0.00,3.9,0.0446,63.6
1,KLK-0002,2024-03-22,LinkedIn,App Install,traffic,25-34,False,NaN,horizontal,True,True,True,True,low,0.07,7433,493,0.0664,3.54,1745.22,34,NaN,0.16,6.4,0.0217,70.0
2,KLK-0003,2024-05-04,TikTok,SaaS,traffic,25-34,False,8.0,vertical,True,True,True,True,medium,0.43,193198,15069,0.0780,0.38,5726.22,492,51053.44,8.92,8.0,0.1000,99.1
3,KLK-0004,2024-12-27,TikTok,Lead Gen,engagement,25-34,False,27.0,quadrado,False,True,True,True,medium,0.54,6506,601,0.0925,0.71,426.71,19,1113.78,2.61,20.3,0.0414,74.9
4,KLK-0005,2024-07-10,Meta,Branding,engagement,25-34,False,34.0,vertical,True,True,True,True,low,0.35,63830,2355,0.0369,0.38,894.90,13,0.00,0.00,34.0,0.0184,77.9


### 1.1. Exemplo de comparação entre grupos

Antes da generalização da lógica de recomendações, foi explorada uma comparação entre campanhas com e sem hook inicial, utilizando o ROAS como métrica.

O exemplo agrupa as campanhas por plataforma e presença de hook, calcula as medianas e compara percentualmente os resultados.

Essa operação ilustra a lógica posteriormente utilizada pelo engine: identificar diferenças de performance entre grupos de criativos que apresentam características distintas.

A comparação, isoladamente, não estabelece que adicionar ou remover um hook seja responsável pela diferença observada.

In [3]:
df_hook = dados.groupby(['platform', 'has_hook'], observed=True)['roas'].agg(['count', 'median'])
com_hook = df_hook.xs(True, level='has_hook')['median']
sem_hook = df_hook.xs(False, level='has_hook')['median']

diferenca_pct = ((com_hook - sem_hook) / sem_hook) * 100
display(diferenca_pct.round(2))
df_hook

platform
LinkedIn     0.00
Meta       -27.07
TikTok      35.91
Name: median, dtype: float64

count  median
platform has_hook               
LinkedIn False        38    0.53
         True         45    0.53
Meta     False       113    1.81
         True        128    1.32
TikTok   False        93    2.20
         True         83    2.99

## 2. Preparação dos dados

Para ampliar as possibilidades de comparação, foram criadas quatro variáveis derivadas.

| Variável | Construção | Finalidade |
|---|---|---|
| `duration_group` | Divisão da duração do vídeo em três faixas por quantis | Comparar durações curtas, médias e longas |
| `audio_group` | Divisão da proporção música/voz em três faixas por quantis | Comparar diferentes composições de áudio |
| `conversion_rate` | Conversões divididas por cliques | Avaliar a eficiência de conversão |
| `watch_time_ratio` | Tempo médio assistido dividido pela duração do vídeo | Avaliar a proporção assistida do criativo |

A divisão por quantis foi utilizada para formar três grupos com quantidades aproximadamente equilibradas de observações, considerando a distribuição do dataset.

Nas razões calculadas, denominadores iguais a zero são substituídos por valores ausentes (`NaN`), evitando divisões inválidas.

As novas variáveis permitem incorporar características de vídeo e métricas derivadas às comparações realizadas pelo sistema.

In [4]:
# Criando grupos de duração para tratar como categórica
dados['duration_group'] = pd.qcut(dados['video_duration_s'], q=3, labels=['short', 'medium', 'long'])

df_duration = dados.groupby(['platform', 'duration_group'], observed=True)['engagement_rate'].agg(['count', 'median'])
display(df_duration)

count   median
platform duration_group                
LinkedIn short               7  0.01250
         medium             20  0.01620
         long               44  0.01820
Meta     short              71  0.01830
         medium             65  0.01660
         long               72  0.01560
TikTok   short              73  0.03750
         medium             56  0.03535
         long               24  0.02245

In [5]:
# Criando grupos de audio para tratar como categórica
dados['audio_group'] = pd.qcut(dados['music_voice_ratio'], q=3, labels=['low_music', 'medium_music', 'high_music'])

df_audio = dados.groupby(['platform', 'audio_group'], observed=True)['engagement_rate'].agg(['count', 'median'])
display(df_audio)

dados['music_voice_ratio'].quantile([0, 1/3, 2/3, 1])

count   median
platform audio_group                 
LinkedIn low_music        27  0.01840
         medium_music     23  0.01230
         high_music       19  0.02280
Meta     low_music        73  0.01640
         medium_music     65  0.02040
         high_music       74  0.01750
TikTok   low_music        51  0.02820
         medium_music     54  0.03005
         high_music       50  0.03755

0.000000    0.02
0.333333    0.29
0.666667    0.49
1.000000    0.92
Name: music_voice_ratio, dtype: float64

In [6]:
# Criando taxa de conversão com base na quantidade de cliques
dados["conversion_rate"] = (dados["conversions"] / dados["clicks"].replace(0, np.nan))

display(dados["conversion_rate"].describe())
print("Valores ausentes:", dados["conversion_rate"].isna().sum())
print("Taxas acima de 100%:", (dados["conversion_rate"] > 1).sum())

count    500.000000
mean       0.034401
std        0.026457
min        0.000000
25%        0.015444
50%        0.030769
75%        0.046921
max        0.156250
Name: conversion_rate, dtype: float64

Valores ausentes: 0
Taxas acima de 100%: 0


In [7]:
# Criando a proporção do vídeo que foi assistida
print(dados[["avg_watch_time_s", "video_duration_s"]].isna().sum())
print("Durações iguais a zero:", (dados["video_duration_s"] == 0).sum())

dados["watch_time_ratio"] = dados["avg_watch_time_s"] / dados["video_duration_s"].replace(0, np.nan)

display(dados["watch_time_ratio"].describe())
print("Valores ausentes:", dados["watch_time_ratio"].isna().sum())
print("Proporções acima de 100%:", (dados["watch_time_ratio"] > 1).sum())

avg_watch_time_s    27
video_duration_s    39
dtype: int64
Durações iguais a zero: 0


count    437.000000
mean       0.542130
std        0.245784
min        0.033333
25%        0.361538
50%        0.508333
75%        0.728571
max        1.000000
Name: watch_time_ratio, dtype: float64

Valores ausentes: 63
Proporções acima de 100%: 0


## 3. Construção do Recommendations Engine

O sistema foi organizado em etapas independentes, responsáveis por identificar oportunidades de melhoria, selecionar recomendações e gerar uma campanha modificada.

### 3.1. Identificação de alternativas

A função `melhor_alternativa()` recebe a plataforma, o atributo analisado, a métrica de performance e o valor atual da campanha.

Para cada comparação:

1. Filtra os dados pela plataforma da campanha.
2. Calcula a mediana da métrica e o número de observações por valor do atributo.
3. Descarta grupos com menos de 10 observações válidas.
4. Seleciona, entre as alternativas ao valor atual, aquela com a maior mediana.
5. Calcula a diferença percentual entre a alternativa e o grupo atual.

Caso não existam grupos suficientes, a mediana atual seja nula ou não positiva, ou nenhuma alternativa apresente resultado superior, a função não gera uma recomendação.

A diferença percentual é calculada por:

$$
\text{Diferença (\%)} = \frac{\text{Mediana alternativa} - \text{Mediana atual}}{\text{Mediana atual}} \times 100
$$

Esse percentual expressa uma diferença relativa entre grupos, e não uma previsão de impacto individual.

In [8]:
def melhor_alternativa(dados, plataforma, atributo, metrica, valor_atual):
    df = dados[dados['platform'] == plataforma]
    resumo = df.groupby(atributo, observed=True)[metrica].agg(['count', 'median'])
    # Considera só grupos com pelo menos 10 campanhas
    resumo = resumo[resumo['count'] >= 10]

    if valor_atual not in resumo.index:
        return None

    mediana_atual = resumo.loc[valor_atual, 'median']
    # Remove a categoria atual das alternativas
    alternativas = resumo.drop(index=valor_atual)

    if alternativas.empty or mediana_atual <= 0:
        return None

    # Seleciona alternativa com maior mediana
    melhor_valor = alternativas['median'].idxmax()
    melhor_mediana = alternativas.loc[melhor_valor, 'median']

    diferenca_pct = (melhor_mediana - mediana_atual) / mediana_atual * 100

    if isinstance(valor_atual, bool) and isinstance(melhor_valor, bool):
        valor_atual = bool(valor_atual)
        melhor_valor = bool(melhor_valor)

    if diferenca_pct <= 0:
        return None

    return {'atributo': atributo, 'valor_atual': valor_atual, 'valor_recomendado': melhor_valor, 
            'metrica': metrica, 'diferenca_pct': float(diferenca_pct), 'mediana_atual': float(mediana_atual),
            'mediana_alternativa': float(melhor_mediana), 'n_atual': int(resumo.loc[valor_atual, 'count']),
            'n_alternativa': int(resumo.loc[melhor_valor, 'count'])}

### 3.2. Geração de recomendações candidatas

A função `gerar_candidatas()` analisa oito atributos modificáveis do criativo:

- Hook inicial, CTA, presença de rosto humano e legendas.
- Densidade de texto e formato do vídeo.
- Grupo de duração e grupo de proporção música/voz.

Cada atributo válido é comparado em relação às cinco métricas de performance utilizadas pelo sistema.

Para incorporar o objetivo da campanha, a função `metrica_por_objetivo()` define a métrica principal:

| Objetivo | Métrica principal |
|---|---|
| Tráfego | CTR |
| Engajamento | Taxa de engajamento |
| Conversões | ROAS |
| Reconhecimento de marca | Proporção assistida do vídeo |
| Instalação de aplicativo | Taxa de conversão |

Essas correspondências são escolhas metodológicas adotadas para o protótipo.

A função ignora atributos ausentes na campanha analisada e registra, para cada candidata válida, a alteração sugerida, a métrica associada, a diferença percentual e os tamanhos dos grupos comparados.

In [9]:
def metrica_por_objetivo(objetivo):
    metricas_por_objetivo = {
    "traffic": "ctr",
    "engagement": "engagement_rate",
    "conversions": "roas",
    "awareness": "watch_time_ratio",
    "app_install": "conversion_rate"}
    return metricas_por_objetivo[objetivo]

def gerar_candidatas(campanha, dados):
    candidatas = []
    # Identificar a métrica principal da campanha
    metrica_principal = metrica_por_objetivo(campanha['objective'])
    metricas = ['ctr', 'conversion_rate', 'roas', 'watch_time_ratio', 'engagement_rate']
    metricas_ordenadas = [metrica_principal] + [m for m in metricas if m != metrica_principal]

    # Percorrer métricas e atributos modificáveis
    atributos = ["has_hook", "has_cta", "has_face", "has_subtitle", "text_density", "duration_group", "audio_group", "format"]
    for atributo in atributos:
        if pd.isna(campanha[atributo]):
            continue
        for metrica in metricas_ordenadas:
            resultado = melhor_alternativa(dados=dados, plataforma=campanha['platform'], atributo=atributo, metrica=metrica, 
                                                         valor_atual=campanha[atributo])
            if resultado is not None:
                resultado["metrica_principal"] = (metrica == metrica_principal)
                candidatas.append(resultado)
    
    return candidatas

### 3.3. Seleção e priorização

A função `priorizar_recomendacoes()` seleciona até cinco sugestões, considerando três critérios principais:

1. **Relevância para o objetivo:** inicialmente, são selecionadas até duas recomendações associadas à métrica principal da campanha.
2. **Diferença mínima:** somente são consideradas recomendações cuja diferença entre medianas seja de pelo menos 10%.
3. **Diversidade de alterações:** um mesmo atributo não pode aparecer mais de uma vez na seleção final.

As vagas restantes são preenchidas com recomendações associadas às demais métricas, quando disponíveis.

Ao final, as recomendações selecionadas são ordenadas pela diferença percentual observada, em ordem decrescente.

Assim, a métrica principal recebe preferência durante a seleção, mas não necessariamente aparece nas primeiras posições da apresentação final.

O limite de 10% é um critério heurístico de seleção adotado para o protótipo, não um limiar de significância estatística.

In [10]:
def priorizar_recomendacoes(candidatas, max_recs=5, diferenca_minima=10):
    recs_finais = []
    
    recs_metrica_principal = [rec for rec in candidatas if rec["metrica_principal"]]
    recs_metrica_principal = sorted(recs_metrica_principal, key=lambda rec: rec["diferenca_pct"], reverse=True)
    
    recs_outras = [rec for rec in candidatas if not rec["metrica_principal"]]
    recs_outras = sorted(recs_outras, key=lambda rec: rec["diferenca_pct"], reverse=True)

    atributos_selecionados = set()

    for rec in recs_metrica_principal:
        if len(recs_finais) >= min(2, max_recs):
            break

        if rec["diferenca_pct"] >= diferenca_minima and rec["atributo"] not in atributos_selecionados:
            recs_finais.append(rec)
            atributos_selecionados.add(rec["atributo"])

    for rec in recs_outras:
        if len(recs_finais) >= max_recs:
            break

        if rec["diferenca_pct"] >= diferenca_minima and rec["atributo"] not in atributos_selecionados:
            recs_finais.append(rec)
            atributos_selecionados.add(rec["atributo"])

    recs_finais = sorted(recs_finais, key=lambda rec: rec["diferenca_pct"], reverse=True)
    return recs_finais

### 3.4. Aplicação das recomendações

A função `aplicar_recomendacoes()` cria uma cópia da campanha original e aplica as alterações selecionadas, sem modificar os dados de entrada.

Para atributos booleanos ou categóricos, o valor atual é substituído pela alternativa recomendada.

Para duração e proporção música/voz, a alternativa inicialmente identificada corresponde a uma faixa categórica. Para fornecer uma sugestão numérica acionável, utiliza-se a mediana da variável original entre campanhas da mesma plataforma pertencentes ao grupo recomendado.

Dessa forma, uma recomendação de duração pode ser apresentada em segundos, enquanto uma recomendação de áudio pode ser expressa como porcentagem de música.

Esses valores representam características típicas do grupo recomendado, não valores ótimos comprovados.

A função retorna a cópia modificada, permitindo comparar as características originais com as sugestões geradas.

In [ ]:
def aplicar_recomendacoes(campanha, recomendacoes, dados):
    campanha_modificada = campanha.copy()
    for rec in recomendacoes:
        atributo = rec["atributo"]

        if atributo in ["duration_group", "audio_group"]:
            coluna_original = {"duration_group": "video_duration_s", "audio_group": "music_voice_ratio"}[atributo]
            valor_recomendado = dados.loc[(dados["platform"] == campanha["platform"]) & (dados[atributo] == rec["valor_recomendado"]), 
                                          coluna_original].median()
            campanha_modificada[coluna_original] = float(valor_recomendado)
            
        campanha_modificada[atributo] = rec["valor_recomendado"]
            
    return campanha_modificada

### 3.5. Integração das etapas

A função `recommendations_engine()` integra a geração de candidatas, a priorização e a aplicação das alterações.

A partir de uma campanha e do dataset preparado, retorna duas estruturas:

- `recomendacoes`: lista com as sugestões selecionadas e suas evidências quantitativas.
- `campanha_modificada`: cópia da campanha contendo as alterações propostas.

Essa separação mantém os resultados estruturados disponíveis para outras formas de apresentação, sem acoplar a lógica de recomendação à interface utilizada pelo avaliador.

In [12]:
def recommendations_engine(campanha, dados):
    candidatas = gerar_candidatas(campanha, dados)
    recomendacoes = priorizar_recomendacoes(candidatas)

    campanha_modificada = aplicar_recomendacoes(campanha, recomendacoes, dados)

    return {"recomendacoes": recomendacoes,"campanha_modificada": campanha_modificada}

## 4. Apresentação das recomendações

Para tornar as sugestões compreensíveis a usuários não técnicos, foi implementada uma camada de apresentação em linguagem natural.

A geração do texto é separada do cálculo das recomendações e utiliza três funções:

- `descrever_acao()`: traduz a alteração proposta em uma instrução específica, considerando o tipo de atributo.
- `descrever_evidencia()`: apresenta a métrica e a diferença percentual observada na plataforma.
- `apresentar_recomendacoes()`: organiza as sugestões e exibe informações da campanha, ações, evidências e ressalva metodológica.

As frases são construídas a partir de modelos determinísticos, sem geração aleatória. Isso permite manter a consistência textual entre diferentes execuções.

A apresentação diferencia as faixas categóricas de duração e áudio dos valores numéricos recomendados, tornando as alterações diretamente compreensíveis.

A ressalva final reforça que as diferenças observadas não representam garantias de melhoria.

In [13]:
def descrever_acao(recomendacao, campanha, campanha_modificada):
    atributo = recomendacao["atributo"]

    acoes_booleanas = {
        "has_hook": ("Adicionar um hook nos primeiros 3 segundos",
                     "Remover o hook inicial"),
        "has_cta": ("Adicionar uma chamada para ação (CTA)",
                    "Remover a chamada para ação (CTA)"),
        "has_subtitle": ("Adicionar legendas ao vídeo",
                         "Remover as legendas do vídeo"),
        "has_face": ("Incluir um rosto humano no vídeo",
                     "Remover a presença de rosto humano")}

    if atributo in acoes_booleanas:
        novo_valor = bool(campanha_modificada[atributo])
        return acoes_booleanas[atributo][0 if novo_valor else 1]
    elif atributo == "format":
        atual = campanha["format"]
        novo = campanha_modificada["format"]
        return f"Alterar o formato de vídeo de {atual} para {novo}"
    elif atributo == "text_density":
        traducoes = {"low": "baixa", "medium": "média", "high": "alta"}
        atual = traducoes.get(campanha["text_density"], campanha["text_density"])
        novo = traducoes.get(campanha_modificada['text_density'], campanha_modificada["text_density"])
        return f"Alterar a densidade de texto de {atual} para {novo}"
    elif atributo == "duration_group":
        atual = campanha["video_duration_s"]
        novo = campanha_modificada["video_duration_s"]
        verbo = "Reduzir" if novo < atual else "Aumentar"
        return (f"{verbo} a duração do vídeo de " f"{atual:.0f}s para aproximadamente {novo:.0f}s")
    elif atributo == "audio_group":
        atual = campanha["music_voice_ratio"] * 100
        novo = campanha_modificada["music_voice_ratio"] * 100
        verbo = "Reduzir" if novo < atual else "Aumentar"
        return (f"{verbo} a proporção de música no áudio " f"de {atual:.0f}% para aproximadamente {novo:.0f}%")
    else:
        raise ValueError(f"Atributo não reconhecido: {atributo}")

In [14]:
def descrever_evidencia(recomendacao, plataforma):
    nomes_metricas = {
    "ctr": "CTR",
    "roas": "ROAS",
    "engagement_rate": "taxa de engajamento",
    "conversion_rate": "taxa de conversão",
    "watch_time_ratio": "proporção assistida do vídeo"}
    metrica = recomendacao["metrica"]
    diferenca = recomendacao["diferenca_pct"]
    nome_metrica = nomes_metricas[metrica]

    return (f"Campanhas com essa característica na plataforma " f"{plataforma} apresentaram {nome_metrica} mediano " 
            f"{diferenca:.1f}% superior ao grupo atual.")

In [15]:
def apresentar_recomendacoes(campanha, resultado):
    recomendacoes = resultado["recomendacoes"]
    campanha_modificada = resultado["campanha_modificada"]

    campanha_id = campanha["campaign_id"]
    plataforma = campanha["platform"]

    objetivos = {
    "awareness": "Reconhecimento de marca",
    "traffic": "Tráfego",
    "conversions": "Conversões",
    "engagement": "Engajamento",
    "app_install": "Instalação de aplicativo"}

    objetivo = objetivos.get(campanha["objective"], campanha["objective"])

    print("=" * 65)
    print(f"RECOMENDAÇÕES PARA A CAMPANHA {campanha_id}")
    print(f"Plataforma: {plataforma} | Objetivo: {objetivo}")
    print("=" * 65)

    if not recomendacoes:
        print("\nNenhuma recomendação válida foi identificada.")
        return
    
    print(f"\n{len(recomendacoes)} recomendações identificadas:\n")

    for i, recomendacao in enumerate(recomendacoes, start=1):
        acao = descrever_acao(recomendacao, campanha, campanha_modificada)
        evidencia = descrever_evidencia(recomendacao, plataforma)
        print(f"{i}. {acao}.")
        print(f"   {evidencia}\n")

    print("-" * 65)
    print("Nota: as recomendações são baseadas em associações observadas no dataset e não garantem melhorias de performance.")

## 5. Testes e validação

A validação do protótipo foi dividida em testes de execução das funções e verificações de propriedades das recomendações.

### 5.1. Teste da geração de recomendações

Inicialmente, uma campanha foi selecionada para inspecionar as candidatas geradas, a priorização e a aplicação das alterações.

O teste permite verificar os atributos selecionados, seus valores atuais e recomendados, as métricas consideradas e as diferenças percentuais observadas.

In [21]:
# CÉLULA DE TESTE --------------------------------------------------------------------------------------------
campanha = dados.iloc[276]

candidatas = gerar_candidatas(campanha, dados)
display(pd.DataFrame(candidatas))
recomendacoes = priorizar_recomendacoes(candidatas)

print(f"Total de recomendações: {len(recomendacoes)}")
display(pd.DataFrame(recomendacoes)[["atributo", "valor_atual", "valor_recomendado", "metrica", "diferenca_pct", "metrica_principal"]])

campanha_modificada = aplicar_recomendacoes(campanha, recomendacoes, dados)
comparacao = pd.DataFrame({"Original": campanha, "Modificada": campanha_modificada})
#display(comparacao)

resultado = recommendations_engine(campanha, dados)
apresentar_recomendacoes(campanha, resultado)

,atributo,valor_atual,valor_recomendado,metrica,diferenca_pct,mediana_atual,mediana_alternativa,n_atual,n_alternativa,metrica_principal
0,has_cta,True,False,watch_time_ratio,0.772947,0.492857,0.496667,96,54,False
1,has_face,True,False,roas,4.101563,2.560000,2.665000,98,78,True
2,has_face,True,False,watch_time_ratio,3.789127,0.481746,0.500000,86,64,False
3,has_subtitle,True,False,conversion_rate,4.911324,0.029482,0.030930,90,70,False
4,has_subtitle,True,False,engagement_rate,19.565217,0.032200,0.038500,83,68,False
5,text_density,low,medium,conversion_rate,4.071634,0.030875,0.032132,45,94,False
6,audio_group,high_music,medium_music,ctr,25.456204,0.054800,0.068750,51,60,False
7,audio_group,high_music,medium_music,conversion_rate,20.342940,0.026383,0.031750,51,60,False
8,audio_group,high_music,medium_music,watch_time_ratio,5.866547,0.483333,0.511688,41,50,False
9,format,vertical,quadrado,roas,0.569260,2.635000,2.650000,152,24,True


Total de recomendações: 2


,atributo,valor_atual,valor_recomendado,metrica,diferenca_pct,metrica_principal
0,audio_group,high_music,medium_music,ctr,25.456204,False
1,has_subtitle,True,False,engagement_rate,19.565217,False


RECOMENDAÇÕES PARA A CAMPANHA KLK-0277
Plataforma: TikTok | Objetivo: Conversões

2 recomendações identificadas:

1. Reduzir a proporção de música no áudio de 59% para aproximadamente 40%.
   Campanhas com essa característica na plataforma TikTok apresentaram CTR mediano 25.5% superior ao grupo atual.

2. Remover as legendas do vídeo.
   Campanhas com essa característica na plataforma TikTok apresentaram taxa de engajamento mediano 19.6% superior ao grupo atual.

-----------------------------------------------------------------
Nota: as recomendações são baseadas em associações observadas no dataset e não garantem melhorias de performance.


### 5.2. Testes automatizados em diferentes campanhas

Foi executada uma bateria de testes com dez campanhas, contemplando diferentes plataformas, objetivos e situações de valores ausentes.

As verificações automatizadas consideraram:

- Execução do engine sem exceções.
- Limite máximo de cinco recomendações.
- Ausência de atributos repetidos nas recomendações selecionadas.
- Diferença percentual mínima de 10% nas sugestões retornadas.

As dez campanhas passaram nas verificações implementadas.

**Escopo da validação:** esses testes confirmam propriedades básicas de execução e seleção, mas não comprovam a qualidade causal das recomendações nem garantem que todos os casos extremos estejam cobertos.

Casos como grupos insuficientes, ausência total de alternativas e menos de três recomendações válidas devem ser considerados separadamente na avaliação de robustez.

In [17]:
# Testes 
campanhas_teste = ["KLK-0017", "KLK-0003", "KLK-0116", "KLK-0010", "KLK-0011", "KLK-0054", "KLK-0001", "KLK-0012", "KLK-0006", "KLK-0009"]

for campanha_id in campanhas_teste:
    campanha = dados.loc[dados["campaign_id"] == campanha_id].iloc[0]

    try:
        resultado = recommendations_engine(campanha, dados)
        recomendacoes = resultado["recomendacoes"]

        atributos = [r["atributo"] for r in recomendacoes]

        assert len(recomendacoes) <= 5
        assert len(atributos) == len(set(atributos))
        assert all(r["diferenca_pct"] >= 10 for r in recomendacoes)

        print(f"OK | {campanha_id} | "f"{len(recomendacoes)} recomendações")

    except Exception as erro:
        print(f"ERRO | {campanha_id} | {erro}")

OK | KLK-0017 | 5 recomendações
OK | KLK-0003 | 4 recomendações
OK | KLK-0116 | 5 recomendações
OK | KLK-0010 | 5 recomendações
OK | KLK-0011 | 5 recomendações
OK | KLK-0054 | 5 recomendações
OK | KLK-0001 | 5 recomendações
OK | KLK-0012 | 5 recomendações
OK | KLK-0006 | 5 recomendações
OK | KLK-0009 | 5 recomendações


### Validação de casos com poucas recomendações

Além dos testes com campanhas de diferentes plataformas e objetivos, o Recommendations Engine foi executado sobre as 500 campanhas do dataset.

Todas foram processadas sem exceções. Dessas, 498 receberam pelo menos três recomendações, enquanto duas (`KLK-0120` e `KLK-0277`) receberam apenas duas.

A análise desses casos mostrou que as alternativas restantes não atendiam ao limite mínimo de 10% de diferença entre medianas ou correspondiam a atributos já selecionados. Optei por preservar os critérios definidos, em vez de gerar recomendações adicionais com evidências mais fracas apenas para atingir a quantidade mínima.

Esse resultado evidencia uma limitação do protótipo e reforça a importância de considerar a disponibilidade de alternativas válidas ao definir a quantidade de recomendações.

## 6. Limitações e conclusão

O Recommendations Engine desenvolvido demonstra como características de criativos podem ser utilizadas para gerar sugestões contextualizadas e quantitativamente fundamentadas.

A solução privilegia interpretabilidade e simplicidade, utilizando comparações de medianas por plataforma, critérios explícitos de seleção e regras de priorização.

### Limitações

**Natureza observacional:** diferenças entre grupos não demonstram que uma alteração específica causará melhoria na performance. Outros fatores das campanhas podem influenciar os resultados.

**Tamanho da amostra:** o dataset contém 500 registros, distribuídos entre diferentes plataformas e características. O mínimo de dez observações por grupo reduz comparações com amostras muito pequenas, mas não elimina a incerteza estatística.

**Contextualização limitada:** as comparações são segmentadas pela plataforma e pelo atributo analisado. Não há controle simultâneo por todas as demais características da campanha.

**Conflitos entre métricas:** uma alteração associada a melhor desempenho em determinada métrica pode não beneficiar outras métricas. A priorização atual não realiza otimização multiobjetivo.

**Integração com o Klike Score:** foi explorada a possibilidade de utilizar o modelo Ridge da Parte 2 para estimar mudanças no score após alterações dos criativos. Entretanto, algumas sugestões favoráveis às métricas de marketing resultavam em reduções do score previsto. Por esse motivo, a simulação não foi incorporada ao fluxo final.

**Valores recomendados:** as durações e proporções de áudio sugeridas são obtidas por medianas dos grupos de referência. Não representam configurações comprovadamente ótimas.

### Conclusão

O protótipo atende ao objetivo de transformar padrões identificados no dataset em recomendações específicas, priorizadas e apresentadas em linguagem acessível.

Sua principal contribuição é oferecer um processo transparente para apoiar decisões de melhoria de criativos, mantendo explícitas as hipóteses e limitações envolvidas.

Evoluções futuras podem incorporar testes controlados, estimativas de incerteza, segmentações mais específicas e estratégias que conciliem métricas de performance com o Klike Score.